# GLM-OCR batch worker (free Colab T4)

Parses every PDF in `Drive/GLM-OCR/inbox` through the full GLM-OCR pipeline
(layout + OCR) on a free T4 and writes results to `Drive/GLM-OCR/outbox`:
- `<name>.md` - markdown with images embedded as base64
- `<name>.json` - layout blocks + usage

**Setup:** Runtime -> Change runtime type -> **T4 GPU**, then Run all.
Stack: vLLM serves `zai-org/GLM-OCR`, glmocr SDK pipeline (layout on GPU),
shim adds the Z.ai-style API + image embedding. Models cache in Drive so
later sessions start fast.

Note: interactive notebook use is fine per Colab ToS; do not expose this as
a public 24/7 API - that is what the cheap VPS + Z.ai API deployment is for.

In [ ]:
!nvidia-smi

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/GLM-OCR'
for d in ('inbox', 'outbox', 'hf_cache'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
os.environ['HF_HOME'] = f'{BASE}/hf_cache'
print('inbox:', len(__import__('pathlib').Path(BASE, 'inbox').glob('*.pdf')), 'pdfs waiting')

In [ ]:
%cd /content
!rm -rf glmocr-box
!git clone https://github.com/idreesmuhammadqazi-create/glmocr-box
%cd glmocr-box
!git pull || true

In [ ]:
# ~5 min: vLLM brings its own torch; glmocr adds the pipeline, shim adds the API
!pip install -q vllm
!pip install -q "glmocr[selfhosted,server]" fastapi "uvicorn[standard]" httpx pymupdf pillow

In [ ]:
import subprocess, time, urllib.request, sys, os

def wait(url, name, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(url, timeout=3)
            print(f'{name} is up ({time.time()-t0:.0f}s)')
            return True
        except Exception:
            time.sleep(5)
    print(f'{name} did NOT come up - check log file')
    return False

# vLLM serving GLM-OCR on the T4 (kills any previous instance first)
subprocess.run('pkill -f "vllm serve"', shell=True)
vllm = subprocess.Popen(
    ['vllm', 'serve', 'zai-org/GLM-OCR', '--port', '8001',
     '--served-model-name', 'glm-ocr', '--max-model-len', '8192',
     '--gpu-memory-utilization', '0.85'],
    stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)
ok1 = wait('http://127.0.0.1:8001/health', 'vllm')
if not ok1:
    print(open('vllm.log').read()[-3000:])

In [ ]:
# glmocr pipeline (layout on the T4) + Z.ai-style shim
env = dict(os.environ,
           GLMOCR_OCR_API_URL='http://127.0.0.1:8001/v1/chat/completions',
           GLMOCR_LAYOUT_DEVICE='cuda',
           GLMOCR_OCR_MODEL='glm-ocr')
subprocess.run('pkill -f glmocr.server', shell=True)
subprocess.run('pkill -f uvicorn', shell=True)
glm = subprocess.Popen([sys.executable, '-m', 'glmocr.server', '--log-level', 'INFO'],
                       env=env, stdout=open('glmocr.log', 'w'), stderr=subprocess.STDOUT)
ok2 = wait('http://127.0.0.1:5002/health', 'glmocr pipeline')
if not ok2:
    print(open('glmocr.log').read()[-3000:])
shim = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'shim.app:app',
                         '--host', '127.0.0.1', '--port', '8000'],
                        stdout=open('shim.log', 'w'), stderr=subprocess.STDOUT)
ok3 = wait('http://127.0.0.1:8000/health', 'shim')
print('ALL READY' if (ok1 and ok2 and ok3) else 'something failed - see logs above')

In [ ]:
# Quick single-file quality check (first 2 pages of first inbox pdf)
import base64, httpx, json, pathlib
pdf = sorted(pathlib.Path(f'{BASE}/inbox').glob('*.pdf'))[0]
b64 = base64.b64encode(pdf.read_bytes()).decode()
r = httpx.post('http://127.0.0.1:8000/paas/v4/layout_parsing',
               json={'model': 'glm-ocr', 'file': b64,
                     'start_page_id': 1, 'end_page_id': 2}, timeout=1800)
r.raise_for_status()
print(r.json()['md_results'][:1500])

In [ ]:
# BATCH: parse everything in inbox -> outbox
import base64, httpx, json, pathlib, time
IN, OUT = pathlib.Path(f'{BASE}/inbox'), pathlib.Path(f'{BASE}/outbox')
pdfs = sorted(IN.glob('*.pdf'))
print(f'{len(pdfs)} pdf(s) to process')
for pdf in pdfs:
    t0 = time.time()
    b64 = base64.b64encode(pdf.read_bytes()).decode()
    try:
        r = httpx.post('http://127.0.0.1:8000/paas/v4/layout_parsing',
                       json={'model': 'glm-ocr', 'file': b64}, timeout=3600)
        r.raise_for_status()
        resp = r.json()
        (OUT / f'{pdf.stem}.md').write_text(resp.get('md_results', ''), encoding='utf-8')
        (OUT / f'{pdf.stem}.json').write_text(
            json.dumps({k: v for k, v in resp.items() if k != 'crop_images'},
                       ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'OK  {pdf.name}  {time.time()-t0:.0f}s  {len(resp.get("md_results",""))} chars')
    except Exception as e:
        print(f'ERR {pdf.name}: {e}')
print('batch done - results in Drive/GLM-OCR/outbox')

## Notes
- First run downloads ~2GB of models to Drive (slow once, fast after).
- Free T4 availability fluctuates; if no GPU is assigned, wait and retry.
- Sessions end after ~12h max / idle timeouts - rerun from the Drive-mount
  cell downward; vLLM + pipeline take ~3-4 min to come back.
- Happy with quality/speed? Next step: 5$/mo no-GPU VPS + this repo's shim
  in `UPSTREAM_MODE=zai` against the Z.ai API (~10k pages per $1).